In [ ]:
import rasterio
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
dist_s1_dir = Path('val_products_transformer_optimized-max10_processed_2026-02-05')

In [ ]:
ts_dirs = sorted(list(dist_s1_dir.glob('*/')))
sample_ts_dir = ts_dirs[1]
sample_ts_dir

In [ ]:
acq_files = sorted(list(sample_ts_dir.rglob('OPERA*_GEN-DIST-STATUS-ACQ.tif')))
acq_files[:3]

In [ ]:
def open_one_arr(path):
    with rasterio.open(path) as src:
        return src.read(1)

def compute_count(ts_dir: Path, glob_pattern: str = '*GEN-DIST-STATUS-ACQ.tif') -> tuple:
    acq_files = sorted(list(ts_dir.rglob(glob_pattern)))
    status_arrs = list(map(open_one_arr, acq_files))
    status_arrs_stacked = np.stack(status_arrs, axis=0)

    valid_observations = (status_arrs_stacked != 255).astype(np.uint8).sum(axis=0)

    with rasterio.open(acq_files[0]) as ds:
        p = ds.profile

    return valid_observations, p,


In [ ]:
interesting_dir = Path('val_products_transformer_optimized-max10_processed_2026-02-05/fire__22LDR')

In [ ]:
valid_observations, p = compute_count(interesting_dir)


In [ ]:
plt.imshow(valid_observations, vmin=0, vmax=70)
plt.colorbar()

In [ ]:
agg_dir_all = Path('aggregated_validation_dist_2024')

In [ ]:
def serialize_count(ts_dir: Path) -> Path:
    agg_dir = agg_dir_all / ts_dir.name
    assert agg_dir.exists(), f"Aggregation directory {agg_dir} does not exist"
    mgrs_tile_id = ts_dir.name.split('__')[-1]
    out_path = agg_dir / f'{mgrs_tile_id}_count_2024.tif'
    arr_c, p_c = compute_count(ts_dir)
    with rasterio.open(out_path, 'w', **p_c) as dst:
        dst.write(arr_c, 1)
    return out_path

serialize_count(sample_ts_dir)



In [40]:
[serialize_count(ts_dir) for ts_dir in tqdm(ts_dirs)]

 12%|█▏        | 10/85 [00:15<01:53,  1.51s/it]


KeyboardInterrupt: 